In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "leeuwen2014human")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "VanLeeuwen_2014_original data exp 1_Ind_vs_Soc study EJC van Leeuwen 2013_2.csv")
complete_path_2 = os.path.join(original_data_pathway, "VanLeeuwen_2014_original data exp 2_Ind_vs_Soc study EJC van Leeuwen 2013_2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

df1['phase']='test_phase_1'
df2['phase']='test_phase_2'

In [3]:
df1.rename(columns={"subject": "ape",
    "condition first": "condition_first",
    "model":"ape_2",
    "order condition": "order_condition",
    "immediate/delayed": "immediate_delayed",
    "first exposure white": "first_exposure_white",
    "first exposure bright": "first_exposure_bright",
    "first exposure brown": "first_exposure_brown",
    "second exposure white": "second_exposure_white",
    "second exposure bright": "second_exposure_bright",
    "second exposure brown": "second_exposure_brown",
    "test first choice": "test_first_choice"}, inplace=True)

In [4]:
df1[['day','month', 'year']] = df1['date'].str.split('/',expand=True)
df1['year'] = '20' + df1['year'].astype(str)

In [5]:
# df2.columns
df2.rename(columns={"subject": "ape", 
    "model":"ape_2",
    "Condition first": "condition_first",
    "order condition": "order_condition",
    "immediate/delayed": "immediate_delayed",
    "first exposure green": "first_exposure_green",
    "first exposure orange": "first_exposure_orange",
    "first exposure blue": "first_exposure_blue",
    "second exposure green": "second_exposure_green",
    "second exposure orange": "second_exposure_orange",
    "second exposure blue": "second_exposure_blue",
    "test first choice": "test_first_choice"}, inplace=True)


In [6]:
df2[['day','month', 'year']] = df2['date'].str.split('/',expand=True)

In [7]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x['study_id']="leeuwen2014human"
    x['role']="focal_participant"
    x['role_2']="model"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)
fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')


In [8]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [9]:
fulldf.dropna(subset=['ape'], inplace=True)
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

import re
replace_1=re.compile('(\,|\?|\.|\!|\:|\;|\')') # / and :
fulldf['comments'].replace(replace_1, '', inplace=True, regex=True)
fulldf['comments'].replace('__', '_', inplace=True, regex=True)

replace_2=re.compile('(\:|\,)') # / and :
fulldf['note'].replace(replace_2, '', inplace=True, regex=True)
fulldf['note'].replace('__', '_', inplace=True, regex=True)



In [10]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
fulldf= fulldf.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    fulldf[x] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
    fulldf[x] = pd.to_datetime(fulldf[x])
    fulldf[y] = pd.to_datetime(fulldf[y])
    fulldf[k] = (fulldf[x] - fulldf[y]).dt.days//365

In [11]:
fulldf=fulldf[['study_id', 'year','month', 'day', 
         'participant','age_in_years','sex', 'role',  
         'participant_2','age_in_years_2', 'sex_2', 'role_2',  'species', 'dyad', 'phase',
        'condition_first', 'order_condition',
        'cup', 'box', 'second', 'cup_2','box_2', 'test_first_choice',  'side',
         'immediate_delayed', 
        #  'first_exposure_white',
      #  'first_exposure_bright', 'first_exposure_brown',
      #  'second_exposure_white', 'second_exposure_bright',
      #  'second_exposure_brown', 'white', 'bright',
      #  'brown',   
      #  'first_exposure_green', 'first_exposure_orange',
      #  'first_exposure_blue', 'second_exposure_green',
      #  'second_exposure_orange', 'second_exposure_blue', 'green', 'orange',
      #  'blue', 
       'comments','note']]


In [12]:
comp_out_path_stand = os.path.join(out_pathway, 'leeuwen2014human_standardized.csv')
fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names = fulldf.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
leeuwen2014human_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'leeuwen2014human_glossary.csv')
leeuwen2014human_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
# for index in range(1,3):
#     exp = fulldf[fulldf['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'leeuwen2014human_exp'+str(index)+'_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'leeuwen2014human_exp'+str(index)+'_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

